# Video preprocessing under compression - Kaggle training
Enable a GPU and Internet, and attach a dataset containing real Kinetics-400 videos in `train/class/video.mp4` and `val/class/video.mp4`.

In [ ]:
!git clone https://github.com/munnn01/preprocessing.git /kaggle/working/preprocessing
%cd /kaggle/working/preprocessing

## Install Kaggle-compatible dependencies
Use `%pip` so packages are installed into this notebook kernel. Dependency warnings for unused preinstalled Kaggle packages can be ignored.

In [ ]:
%pip install -q --no-cache-dir -r requirements-kaggle.txt

In [ ]:
import compressai
import cv2
import numpy
import torch
import torchvision

print("NumPy:", numpy.__version__)
print("OpenCV:", cv2.__version__)
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CompressAI:", compressai.__version__)
print("CUDA:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Enable a GPU in Kaggle Notebook settings"

In [ ]:
from pathlib import Path

DATA = Path("/kaggle/input/kinetics-train-5per/kinetics400_5per/kinetics400_5per")
assert (DATA / "train").is_dir(), f"Missing train split: {DATA / 'train'}"
if not (DATA / "val").is_dir():
    print("No val/ directory: train.py will create a stratified 80/20 split.")
print(DATA)

## Smoke test
This downloads the pretrained weights and validates the complete differentiable path on a few videos.

In [ ]:
!python -u train.py --data-root "{DATA}" --val-ratio 0.2 --codec-qualities 3 --batch-size 2 --workers 4 --smoke-test --output-dir /kaggle/working/checkpoints-smoke

## Full run
The effective batch is four clips through gradient accumulation. SSF2020 quality is sampled from 1, 3, and 5 once per epoch.

In [ ]:
!python -u train.py --data-root "{DATA}" --val-ratio 0.2 --epochs 30 --frames 16 --frame-stride 2 --frame-size 128 --temporal-frames 8 --codec-qualities 1 3 5 --batch-size 2 --accumulation-steps 4 --workers 4 --amp --output-dir /kaggle/working/checkpoints

## Visualize the learned pipeline
This writes a frame grid, comparison MP4, and JSON metrics for one validation clip.

In [ ]:
!python -u visualize_pipeline.py --checkpoint /kaggle/working/checkpoints/best.pt --data-root "{DATA}" --split val --sample-index 0 --codec-quality 3 --show-frames 4 --output-dir /kaggle/working/visualization
from IPython.display import Image, Video, display

display(Image("/kaggle/working/visualization/pipeline-comparison.png"))
display(Video("/kaggle/working/visualization/pipeline-comparison.mp4", embed=True))

## Real-codec smoke evaluation

In [ ]:
!ffmpeg -hide_banner -encoders 2>&1 | grep -E 'libx264|libx265'
!python -u evaluate_real_codec.py --checkpoint /kaggle/working/checkpoints/best.pt --data-root "{DATA}" --split val --codecs h264 h265 --qps 30 35 40 45 50 --limit 10 --output-dir /kaggle/working/real-codec-smoke